# Crop mapping and economic value estimation: Tadla irrigated perimeter (Morocco)

Sentinel-2 monthly NDVI time series, Random Forest classification and a simple economic valuation, for the 2024-2025 agricultural season.

**How to run:** open in Google Colab, set `GCP_PROJECT` below, then *Runtime > Run all*. Training points are read from `data/train_points_v2.geojson`.

## 0. Setup

In [ ]:
%pip install -q earthengine-api geemap geopandas scikit-learn

In [ ]:
import os
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Replace with your own Google Cloud project ID registered for Earth Engine
GCP_PROJECT = "YOUR_GCP_PROJECT_ID"

try:
    ee.Initialize(project=GCP_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GCP_PROJECT)

# Study area [west, south, east, north] and season
AOI = ee.Geometry.Rectangle([-6.75, 32.25, -6.30, 32.43])
START, END = "2024-09-01", "2025-08-31"

# Class definitions (class 2, sugar beet, was not modelled)
NAMES = {1: "Cereals", 3: "Alfalfa", 4: "Tree crops", 5: "Market gardening",
         6: "Bare soil", 7: "Built-up / other"}
PALETTE = ["#f4d03f", "#e67e22", "#27ae60", "#145a32",
           "#8e44ad", "#b9770e", "#7f8c8d"]   # index = class - 1

for d in ["data", "results/maps", "results/figures"]:
    os.makedirs(d, exist_ok=True)

## 1. Sentinel-2 monthly NDVI composites

Cloud and shadow masking with the SCL band, then NDVI and a median composite per month (12 bands).

In [ ]:
def mask_scl(img):
    scl = img.select("SCL")
    ok = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
             .And(scl.neq(10)).And(scl.neq(11)))
    return img.updateMask(ok)

def add_ndvi(img):
    b = img.select(["B4", "B8"]).divide(10000)
    ndvi = b.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return ndvi.copyProperties(img, ["system:time_start"])

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(AOI).filterDate(START, END)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
        .map(mask_scl).map(add_ndvi))
print("Scenes:", s2.size().getInfo())

def monthly(m):
    start = ee.Date(START).advance(m, "month")
    end = start.advance(1, "month")
    return (s2.filterDate(start, end).median()
              .set("system:index", start.format("YYYY_MM"))
              .set("system:time_start", start.millis()))

monthly_col = ee.ImageCollection(ee.List.sequence(0, 11).map(monthly))
ndvi_stack = monthly_col.select("NDVI").toBands().clip(AOI)
print(ndvi_stack.bandNames().getInfo())

## 2. Training points

Points were interpreted visually (high-resolution imagery and NDVI profiles), not collected in the field. They are stored in `data/train_points_v2.geojson` with a `classe` property.

In [ ]:
POINTS_FILE = "data/train_points_v2.geojson"
POINTS_ASSET = f"projects/{GCP_PROJECT}/assets/train_points_v2"

if os.path.exists(POINTS_FILE):
    pts = geemap.geojson_to_ee(POINTS_FILE)
else:
    # first run by the author: read the Earth Engine asset and keep a copy for the repository
    pts = ee.FeatureCollection(POINTS_ASSET)
    geemap.ee_to_gdf(pts).to_file(POINTS_FILE, driver="GeoJSON")

pts = pts.map(lambda f: f.set("classe", ee.Number(f.get("classe")).toInt()))
print(pts.aggregate_histogram("classe").getInfo())

## 3. Sampling and validation

NDVI values are extracted under each point. A stratified 5-fold cross-validation estimates accuracy. With so few points the estimate is noisy and probably optimistic (points are spatially clustered).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

samples = ndvi_stack.sampleRegions(collection=pts, properties=["classe"], scale=10)
df = geemap.ee_to_df(samples)
print(df.shape)

cols = [c for c in df.columns if c.endswith("_NDVI")]
X, y = df[cols], df["classe"].astype(int)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pred = cross_val_predict(
    RandomForestClassifier(n_estimators=300, random_state=42), X, y, cv=cv)

labels = sorted(y.unique())
print("Cross-validated accuracy:", round(accuracy_score(y, pred), 2))
print(classification_report(y, pred, labels=labels,
                            target_names=[NAMES[l] for l in labels]))
print(pd.DataFrame(confusion_matrix(y, pred, labels=labels),
                   index=[NAMES[l] for l in labels],
                   columns=[NAMES[l] for l in labels]))

## 4. Classification map

Random Forest trained on all points, applied to every pixel, then smoothed with a 3x3 majority filter.

In [ ]:
clf = ee.Classifier.smileRandomForest(300).train(
    features=samples, classProperty="classe",
    inputProperties=ndvi_stack.bandNames())

classified = ndvi_stack.classify(clf)
smooth = classified.focalMode(radius=1, kernelType="square", units="pixels")

vis = {"min": 1, "max": 7, "palette": PALETTE}

m = geemap.Map()
m.add_basemap("SATELLITE")
m.centerObject(AOI, 10)
m.addLayer(smooth, {**vis, "opacity": 0.6}, "Crop map")
m

In [ ]:
import requests

url = smooth.visualize(**vis).getThumbURL(
    {"region": AOI, "dimensions": 1600, "format": "png"})
with open("results/maps/crop_map.png", "wb") as f:
    f.write(requests.get(url).content)
print("Saved results/maps/crop_map.png")

## 5. Area per class

Areas are computed on the smoothed map at 30 m resolution, so they are orders of magnitude, not survey statistics.

In [ ]:
area = ee.Image.pixelArea().divide(10000).addBands(smooth.rename("classe"))
stats = area.reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName="classe"),
    geometry=AOI, scale=30, maxPixels=1e10, tileScale=4)

surf = pd.DataFrame(ee.List(stats.get("groups")).getInfo())
surf["classe"] = surf["classe"].astype(int)
surf["culture"] = surf["classe"].map(NAMES)
surf["ha"] = surf["sum"].round(0)
surf["%"] = (100 * surf["sum"] / surf["sum"].sum()).round(1)
surf = surf.sort_values("classe")[["classe", "culture", "ha", "%"]]
surf.to_csv("results/area_by_class.csv", index=False)

ax = surf.set_index("culture")["ha"].plot.barh(
    color=[PALETTE[c - 1] for c in surf["classe"]], figsize=(7, 4))
ax.set_xlabel("Area (ha)")
plt.tight_layout()
plt.savefig("results/figures/area_by_class.png", dpi=200)
surf

## 6. Economic valuation (gross revenue)

Gross revenue = area x yield x price, under three scenarios (low, central, high). Parameters and sources are listed in `data/parametres_economiques.csv`. Empty cells mean no reliable source was found, so those classes are not valued. Production costs are not included, so this is **not** a margin.

In [ ]:
params = pd.DataFrame([
  dict(classe=1, culture="Cereals",
       rend_bas=3.5, rend_central=4.0, rend_haut=4.4,
       prix_bas=2800, prix_central=2800, prix_haut=2800,
       source="Yield: La Vie eco (irrigated 35-40 q/ha); ORMVA Haouz 2016-17 (44 q/ha, other perimeter). Price: soft wheat reference price 280 Dh/q (ONICL, 2026 campaign)"),
  dict(classe=3, culture="Alfalfa",
       rend_bas=None, rend_central=None, rend_haut=None,
       prix_bas=None, prix_central=None, prix_haut=None,
       source="to be completed"),
  dict(classe=4, culture="Tree crops (olive proxy)",
       rend_bas=2.2, rend_central=3.2, rend_haut=3.9,
       prix_bas=4000, prix_central=5000, prix_haut=6000,
       source="Yield: ORMVAT (3.2 t/ha), 2.2 t/ha irrigated 2024-25, about 3.9 t/ha normal year (computed). Olive price 4-6 Dh/kg (Bladi, Feb 2026)"),
  dict(classe=5, culture="Market gardening",
       rend_bas=None, rend_central=None, rend_haut=None,
       prix_bas=None, prix_central=None, prix_haut=None,
       source="to be completed"),
])
params.to_csv("data/parametres_economiques.csv", index=False)

eco = surf.drop(columns=["culture"]).merge(
    params.drop(columns=["culture", "source"]), on="classe", how="left")
eco["culture"] = eco["classe"].map(NAMES)
for s in ["bas", "central", "haut"]:
    eco[f"revenue_{s}_MDh"] = (
        eco["ha"] * eco[f"rend_{s}"] * eco[f"prix_{s}"] / 1e6).round(1)

eco = eco[["culture", "ha", "revenue_bas_MDh", "revenue_central_MDh", "revenue_haut_MDh"]]
eco.to_csv("results/economic_estimates.csv", index=False)
eco

## Limitations

- Training labels come from visual interpretation, not field data; accuracy is estimated on few, clustered points.
- Sugar beet is not modelled and is probably absorbed by other classes (mostly cereals).
- Natural steppe and rangeland have no class and fall into "bare soil" or "built-up / other".
- Tree crops mix olive and citrus but are valued as olive.
- Revenue only: production costs are missing, and alfalfa and market gardening are not valued.
- Yields are reference values from reports, not estimated from imagery.